In [1]:
import pandas as pd
import numpy as np

# Препроцессинг и пайплайны
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Модели
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Валидация и подбор гиперпараметров
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [2]:
import io
import requests

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'

# Скачиваем через requests (отключая строгую проверку ssl при необходимости)
response = requests.get(url, verify=False)  # verify=False проигнорирует SSL-ошибки
df = pd.read_csv(io.StringIO(response.text))

df.head()

C:\Users\sirot\mlfirst+\.venv\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
df['Churn'].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [5]:
print(df['TotalCharges'])

0         29.85
1        1889.5
2        108.15
3       1840.75
4        151.65
         ...   
7038     1990.5
7039     7362.9
7040     346.45
7041      306.6
7042     6844.5
Name: TotalCharges, Length: 7043, dtype: str


In [6]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = X.drop(columns=numeric_features).columns.tolist()
y = (df["Churn"] == "Yes").astype(int)

X.shape

(7043, 19)

In [10]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onebot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ]
)

X_transformed = preprocessor.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

(X_transformed).shape

(7043, 46)

In [14]:
pipe_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
])

pipe_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

cv_scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring='accuracy')
cv_scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=cv, scoring='accuracy')

print('Фолды RF:', np.round(cv_scores_rf, 4))
print(f"Средняя точность RF: {cv_scores_rf.mean():.4f} +- {cv_scores_rf.std():.4f}")
print(f"Средняя точность LR: {cv_scores_lr.mean():.4f} ± {cv_scores_lr.std():.4f}")
diff = cv_scores_rf.mean() - cv_scores_lr.mean()
print(f"Разница (RF - LR): {diff:+.4f}")

Фолды RF: [0.7844 0.7888 0.7995 0.8048 0.8037]
Средняя точность RF: 0.7962 +- 0.0082
Средняя точность LR: 0.8026 ± 0.0120
Разница (RF - LR): -0.0064


In [16]:
pipe_lr.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)

y_pred_rf = pipe_rf.predict(X_test)
y_prob_rf = pipe_rf.predict_proba(X_test)[:,1]

y_pred_lr = pipe_lr.predict(X_test)
y_prob_lr = pipe_lr.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "Метрика":["Accuracy", "F1-Score", "ROC-AUC"],
    "Logistic Regression": [
        accuracy_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr),
        roc_auc_score(y_test, y_prob_lr)
    ],
    'Random Forest': [
        accuracy_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf),
        roc_auc_score(y_test, y_prob_rf)
    ]
})

print('РЕЗУЛЬТАТЫ НА ОТЛОЖЕННОМ ТЕСТЕ (X_test):')
print(results.round(4).to_string(index=False))

РЕЗУЛЬТАТЫ НА ОТЛОЖЕННОМ ТЕСТЕ (X_test):
 Метрика  Logistic Regression  Random Forest
Accuracy               0.8055         0.7935
F1-Score               0.6040         0.5237
 ROC-AUC               0.8419         0.8397


In [22]:
from lightgbm import LGBMClassifier
pipe_lgb = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(random_state=42, verbose=-1))
])

param_grid_lr = {
    'model__C': [0.01,0.1,1.0,10.0],
    'model__class_weight': [None, 'balanced']
}
grid_lr=GridSearchCV(pipe_lr, param_grid_lr, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_lr.fit(X_train, y_train)

param_grid_lgb = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.03, 0.1],
    "model__scale_pos_weight": [1, 2.5] 
}

grid_lgb = GridSearchCV(pipe_lgb, param_grid_lgb, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_lgb.fit(X_train, y_train)

print(f"Лучший ROC-AUC на CV (Logistic Regression): {grid_lr.best_score_:.4f}")
print("Параметры LR:", grid_lr.best_params_)
print(f"\nЛучший ROC-AUC на CV (LightGBM): {grid_lgb.best_score_:.4f}")
print("Параметры LGBM:", grid_lgb.best_params_)

Лучший ROC-AUC на CV (Logistic Regression): 0.8463
Параметры LR: {'model__C': 10.0, 'model__class_weight': None}

Лучший ROC-AUC на CV (LightGBM): 0.8487
Параметры LGBM: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 50, 'model__scale_pos_weight': 1}


In [25]:
from catboost import CatBoostClassifier

# 1. Готовим данные: заполняем пропуски в категориальных (если есть) и приводим к str
X_train_cb = X_train.copy()
X_train_cb[categorical_features] = X_train_cb[categorical_features].fillna("Missing").astype(str)

# 2. Инициализируем модель БЕЗ cat_features в конструкторе
cb_model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.05,
    depth=4,
    eval_metric="AUC",
    random_seed=42,
    verbose=0
)

# 3. Кросс-валидация через ручной цикл (самый чистый способ для CatBoost, без багов sklearn.clone)
cv_scores_cb = []
for train_idx, val_idx in cv.split(X_train_cb, y_train):
    X_f_train, X_f_val = X_train_cb.iloc[train_idx], X_train_cb.iloc[val_idx]
    y_f_train, y_f_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    cb_model.fit(
        X_f_train, y_f_train,
        cat_features=categorical_features,
        eval_set=(X_f_val, y_f_val),
        verbose=0
    )
    y_prob = cb_model.predict_proba(X_f_val)[:, 1]
    cv_scores_cb.append(roc_auc_score(y_f_val, y_prob))

cv_scores_cb = np.array(cv_scores_cb)

print("Фолды CatBoost ROC-AUC:", np.round(cv_scores_cb, 4))
print(f"Средний ROC-AUC CatBoost: {cv_scores_cb.mean():.4f} ± {cv_scores_cb.std():.4f}")
print(f"Средний ROC-AUC LightGBM: {grid_lgb.best_score_:.4f}")
print(f"Средний ROC-AUC LogReg:   {grid_lr.best_score_:.4f}")

Фолды CatBoost ROC-AUC: [0.845  0.833  0.848  0.8659 0.861 ]
Средний ROC-AUC CatBoost: 0.8506 ± 0.0118
Средний ROC-AUC LightGBM: 0.8487
Средний ROC-AUC LogReg:   0.8463


In [28]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

X_test_cb = X_test.copy()
X_test_cb[categorical_features] = X_test_cb[categorical_features].fillna('Missing').astype(str)

cb_model.fit(X_train_cb, y_train, cat_features=categorical_features, verbose=0)

y_test_probs = cb_model.predict_proba(X_test_cb)[:, 1]

thresholds = np.arange(0.1, 0.75, 0.05)
threshold_results = []

for t in thresholds:
    y_pred_t = (y_test_probs >=t).astype(int)

    threshold_results.append({
        "Порог (Threshold)": round(t, 2),
        "Precision": round(precision_score(y_test, y_pred_t, zero_division=0), 4),
        "Recall (Полнота)": round(recall_score(y_test, y_pred_t), 4),
        "F1-Score": round(f1_score(y_test, y_pred_t), 4)
    })

df_thresholds = pd.DataFrame(threshold_results)
print(df_thresholds.to_string(index=False))

 Порог (Threshold)  Precision  Recall (Полнота)  F1-Score
              0.10     0.4062            0.9492    0.5689
              0.15     0.4379            0.8957    0.5882
              0.20     0.4738            0.8476    0.6079
              0.25     0.5033            0.8075    0.6201
              0.30     0.5289            0.7594    0.6235
              0.35     0.5730            0.7139    0.6357
              0.40     0.6020            0.6551    0.6274
              0.45     0.6398            0.5936    0.6158
              0.50     0.6838            0.5321    0.5985
              0.55     0.7000            0.4305    0.5331
              0.60     0.7216            0.3396    0.4618
              0.65     0.7698            0.2861    0.4172
              0.70     0.7944            0.2273    0.3534


In [29]:
import joblib

# Сохраняем готовую обученную модель CatBoost
joblib.dump(cb_model, "churn_catboost_model.joblib")
print("Модель успешно сохранена в churn_catboost_model.joblib!")

Модель успешно сохранена в churn_catboost_model.joblib!


In [30]:
from sklearn.metrics import roc_auc_score, f1_score

# 1. Получаем вероятности от каждой модели на отложенном тесте
prob_lr = grid_lr.best_estimator_.predict_proba(X_test)[:, 1]
prob_lgb = grid_lgb.best_estimator_.predict_proba(X_test)[:, 1]
prob_cb = cb_model.predict_proba(X_test_cb)[:, 1]

# 2. Смешиваем вероятности (взвешенный блендинг)
prob_blend = 0.50 * prob_cb + 0.30 * prob_lgb + 0.20 * prob_lr

# 3. Считаем ROC-AUC для всех участников
auc_lr = roc_auc_score(y_test, prob_lr)
auc_lgb = roc_auc_score(y_test, prob_lgb)
auc_cb = roc_auc_score(y_test, prob_cb)
auc_blend = roc_auc_score(y_test, prob_blend)

# 4. Проверим F1-score при оптимальном пороге 0.35
f1_lr = f1_score(y_test, (prob_lr >= 0.35).astype(int))
f1_lgb = f1_score(y_test, (prob_lgb >= 0.35).astype(int))
f1_cb = f1_score(y_test, (prob_cb >= 0.35).astype(int))
f1_blend = f1_score(y_test, (prob_blend >= 0.35).astype(int))

# 5. Выводим итоговое сравнение
blend_results = pd.DataFrame({
    "Модель / Ансамбль": [
        "Logistic Regression", 
        "LightGBM", 
        "CatBoost", 
        "🏆 БЛЕНД (CatBoost + LGBM + LR)"
    ],
    "ROC-AUC на тесте": [auc_lr, auc_lgb, auc_cb, auc_blend],
    "F1-Score (порог 0.35)": [f1_lr, f1_lgb, f1_cb, f1_blend]
})

print(blend_results.round(4).to_string(index=False))

             Модель / Ансамбль  ROC-AUC на тесте  F1-Score (порог 0.35)
           Logistic Regression            0.8411                 0.6179
                      LightGBM            0.8471                 0.6407
                      CatBoost            0.8463                 0.6357
🏆 БЛЕНД (CatBoost + LGBM + LR)            0.8473                 0.6344
